In [1]:
# ============================================================================
# NOTEBOOK 03 — FEATURE ENGINEERING SGCC
# ============================================================================
# Objetivo: A partir del Parquet limpio (32M registros, formato largo),
# calcular 14 features estadísticas por cliente y exportar el CSV final
# que se cargará en RapidMiner para el clasificador.
#
# Features a calcular (14):
#   Estadísticas globales: media, std, max, min, mediana
#   Forma:                 coef_variacion, skewness, kurtosis
#   Anomalías:             dias_consumo_cero, racha_max_ceros, n_picos, n_valles
#   Comportamiento:        pct_dias_bajo_media, tendencia_lineal
# ============================================================================

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType, IntegerType

spark = (
    SparkSession.builder
    .appName("TFM-SGCC-Features")
    .master("spark://spark-master:7077")
    .config("spark.executor.memory", "1500m")
    .config("spark.driver.memory", "1500m")
    .config("spark.sql.shuffle.partitions", "16")
    .config("mapreduce.fileoutputcommitter.algorithm.version", "2")
    .config("spark.hadoop.mapreduce.fileoutputcommitter.algorithm.version", "2")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} listo")

Spark 3.5.0 listo


In [ ]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 5)

In [ ]:
RUTA_PROCESSED = "/opt/spark-data/processed/sgcc_clean.parquet"
df = spark.read.parquet(RUTA_PROCESSED)

print("Esquema:")
df.printSchema()

print("\nPrimeras filas:")
df.show(3)

# Estimación rápida de clientes (orden de magnitud), evita el distinct().count()
n_clientes_aprox = df.select(F.approx_count_distinct("CONS_NO")).collect()[0][0]
print(f"\nClientes únicos (aproximado): {n_clientes_aprox:,}")
print("(debería rondar 31.188 según el notebook 02)")

Esquema:
root
 |-- CONS_NO: string (nullable = true)
 |-- FLAG: integer (nullable = true)
 |-- fecha: date (nullable = true)
 |-- consumo: double (nullable = true)


Primeras filas:
+--------------------+----+----------+-------+
|             CONS_NO|FLAG|     fecha|consumo|
+--------------------+----+----------+-------+
|09D543A0C34BC4614...|   0|2014-06-08|  16.54|
|28AFEF32E8E5CBFC9...|   0|2016-10-21|    0.0|
|33BF72E748DB8F1C9...|   0|2014-07-05|   4.82|
+--------------------+----+----------+-------+
only showing top 3 rows


Clientes únicos (aproximado): 33,285
(debería rondar 31.188 según el notebook 02)


In [ ]:
# Añadimos la media del cliente como columna usando una función de ventana

ventana_cli = Window.partitionBy("CONS_NO")

df_enriquecido = df.withColumn(
    "media_cli",
    F.mean("consumo").over(ventana_cli)
)

df_enriquecido.cache()

df_enriquecido.take(1)
print("✓ DataFrame con media por cliente cacheado")

✓ DataFrame con media por cliente cacheado


In [ ]:
# Agregación única que calcula 12 de las 14 features de golpe
df_agg = df_enriquecido.groupBy("CONS_NO").agg(
    F.first("FLAG").alias("FLAG"),
    
    # Estadísticas globales (5 features)
    F.mean("consumo").alias("media"),
    F.stddev("consumo").alias("std"),
    F.max("consumo").alias("max"),
    F.min("consumo").alias("min"),
    F.expr("percentile_approx(consumo, 0.5)").alias("mediana"),
    
    # Forma (2 features) — la coef_variacion la calculamos después como std/media
    F.skewness("consumo").alias("skewness"),
    F.kurtosis("consumo").alias("kurtosis"),
    
    # Anomalías — días con consumo = 0
    F.sum(F.when(F.col("consumo") == 0, 1).otherwise(0)).alias("dias_consumo_cero"),
    
    # Picos y valles (usan media_cli que ya tenemos)
    F.sum(F.when(F.col("consumo") > 3 * F.col("media_cli"), 1).otherwise(0)).alias("n_picos"),
    F.sum(F.when(
        (F.col("consumo") > 0) & (F.col("consumo") < 0.3 * F.col("media_cli")), 1
    ).otherwise(0)).alias("n_valles"),
    
    # % bajo la media del cliente
    F.mean(F.when(F.col("consumo") < F.col("media_cli"), 1.0).otherwise(0.0))
     .alias("pct_dias_bajo_media"),
)

print("✓ Agregación única ejecutada")
df_agg.show(3)

✓ Agregación única ejecutada
+--------------------+----+------------------+-------------------+----+----+-------+------------------+------------------+-----------------+-------+--------+-------------------+
|             CONS_NO|FLAG|             media|                std| max| min|mediana|          skewness|          kurtosis|dias_consumo_cero|n_picos|n_valles|pct_dias_bajo_media|
+--------------------+----+------------------+-------------------+----+----+-------+------------------+------------------+-----------------+-------+--------+-------------------+
|00150F4BEFB093DD8...|   0|0.9646228239845357|0.08277556226905584|1.93|0.48|   0.94|1.0531121095189722| 24.32904178300195|                0|      0|       0|  0.632495164410058|
|004CD28631398E708...|   0|1.3401740812379095| 0.8524644352830423|4.92|0.07|   1.09|0.9756808184702783|0.3327368120567278|                0|      6|      32| 0.5889748549323017|
|006A4BFF393612BB0...|   0|2.0889168278529957| 0.7950777826063232|5.81|0.45|   1.

In [5]:
# La tendencia es la pendiente de la regresión lineal consumo ~ dia
# La calculamos como cov(x,y) / var(x)
# Hacemos otra agregación porque necesita la fecha convertida a número

df_tendencia = (
    df.withColumn("dia_num", F.datediff(F.col("fecha"), F.lit("2014-01-01")))
      .groupBy("CONS_NO")
      .agg(
          (F.covar_pop("dia_num", "consumo") / F.var_pop("dia_num"))
            .alias("tendencia_lineal")
      )
)

print("✓ Tendencia lineal calculada")
df_tendencia.show(3)

✓ Tendencia lineal calculada
+--------------------+--------------------+
|             CONS_NO|    tendencia_lineal|
+--------------------+--------------------+
|772ECBB72F72DA94C...|0.003701575085536...|
|F7F83C0E5772DF544...|0.002356149305107...|
|2CCBFBC365F4A9AEC...|0.007406423557409832|
+--------------------+--------------------+
only showing top 3 rows



In [6]:
# Marcamos consumo=0 con 1, detectamos cambios de estado y formamos grupos de racha
ventana_ord = Window.partitionBy("CONS_NO").orderBy("fecha")
ventana_acum = ventana_ord.rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_rachas_raw = (
    df.select("CONS_NO", "fecha", "consumo")
      .withColumn("es_cero", F.when(F.col("consumo") == 0, 1).otherwise(0))
      .withColumn(
          "cambio",
          F.when(F.lag("es_cero").over(ventana_ord) != F.col("es_cero"), 1).otherwise(0)
      )
      .withColumn("grupo_racha", F.sum("cambio").over(ventana_acum))
)

# Solo nos quedamos con las filas donde es_cero=1, agrupamos por (CONS_NO, grupo)
# y nos quedamos con la racha más larga por cliente
df_racha_max = (
    df_rachas_raw
    .filter(F.col("es_cero") == 1)
    .groupBy("CONS_NO", "grupo_racha")
    .count()
    .groupBy("CONS_NO")
    .agg(F.max("count").alias("racha_max_ceros"))
)

print("✓ Racha máxima de ceros calculada")
df_racha_max.show(3)

✓ Racha máxima de ceros calculada
+--------------------+---------------+
|             CONS_NO|racha_max_ceros|
+--------------------+---------------+
|008D2EB6431864326...|             69|
|008F173DBB054582C...|              1|
|00999F7EF358E158C...|              1|
+--------------------+---------------+
only showing top 3 rows



In [ ]:
df_features = (
    df_agg
    .join(df_tendencia, on="CONS_NO", how="left")
    .join(df_racha_max, on="CONS_NO", how="left")
    # Coeficiente de variación = std/media (evitando división por cero)
    .withColumn(
        "coef_variacion",
        F.when(F.col("media") > 0, F.col("std") / F.col("media")).otherwise(0.0)
    )
    # Rellenamos nulos en clientes que no tuvieron rachas de ceros
    .fillna({
        "racha_max_ceros": 0,
        "skewness": 0.0,
        "kurtosis": 0.0,
        "tendencia_lineal": 0.0,
    })
)

columnas_orden = [
    "CONS_NO", "FLAG",
    "media", "std", "max", "min", "mediana",
    "coef_variacion", "skewness", "kurtosis",
    "dias_consumo_cero", "racha_max_ceros", "n_picos", "n_valles",
    "pct_dias_bajo_media", "tendencia_lineal",
]
df_features = df_features.select(columnas_orden)

print(f"✓ Tabla final: {df_features.count():,} clientes × {len(df_features.columns)} columnas")
df_features.show(5)

✓ Tabla final: 31,188 clientes × 16 columnas
+--------------------+----+------------------+-------------------+-----+----+-------+-------------------+------------------+-------------------+-----------------+---------------+-------+--------+-------------------+--------------------+
|             CONS_NO|FLAG|             media|                std|  max| min|mediana|     coef_variacion|          skewness|           kurtosis|dias_consumo_cero|racha_max_ceros|n_picos|n_valles|pct_dias_bajo_media|    tendencia_lineal|
+--------------------+----+------------------+-------------------+-----+----+-------+-------------------+------------------+-------------------+-----------------+---------------+-------+--------+-------------------+--------------------+
|00150F4BEFB093DD8...|   0|0.9646228239845357|0.08277556226905584| 1.93|0.48|   0.94|0.08581132460368039|1.0531121095189722|  24.32904178300195|                0|              0|      0|       0|  0.632495164410058|-2.01641048758363...|
|004CD2

In [ ]:
df_features.describe().show()

+-------+--------------------+------------------+------------------+-----------------+-----------------+------------------+------------------+------------------+-------------------+------------------+------------------+------------------+------------------+-----------------+-------------------+--------------------+
|summary|             CONS_NO|              FLAG|             media|              std|              max|               min|           mediana|    coef_variacion|           skewness|          kurtosis| dias_consumo_cero|   racha_max_ceros|           n_picos|         n_valles|pct_dias_bajo_media|    tendencia_lineal|
+-------+--------------------+------------------+------------------+-----------------+-----------------+------------------+------------------+------------------+-------------------+------------------+------------------+------------------+------------------+-----------------+-------------------+--------------------+
|  count|               31188|             31188|

In [ ]:
df_pandas = df_features.toPandas()

print(f"DataFrame Pandas: {df_pandas.shape[0]:,} filas × {df_pandas.shape[1]} columnas")
print(f"Memoria ocupada: {df_pandas.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Guardamos con Pandas (sin Spark, sin bug del rename)
RUTA_CSV_FINAL = "/opt/spark-data/features/sgcc_features.csv"

import os
os.makedirs(os.path.dirname(RUTA_CSV_FINAL), exist_ok=True)
df_pandas.to_csv(RUTA_CSV_FINAL, index=False)

size_mb = os.path.getsize(RUTA_CSV_FINAL) / 1024**2
print(f"\n✓ CSV guardado: {RUTA_CSV_FINAL}")
print(f"✓ Tamaño: {size_mb:.2f} MB")

DataFrame Pandas: 31,188 filas × 16 columnas
Memoria ocupada: 6.10 MB

✓ CSV guardado: /opt/spark-data/features/sgcc_features.csv
✓ Tamaño: 5.68 MB


In [15]:
import pandas as pd

df_check = pd.read_csv(RUTA_CSV_FINAL)
print(f"Verificación lectura: {df_check.shape[0]:,} filas × {df_check.shape[1]} columnas")
print("\nDistribución FLAG:")
print(df_check["FLAG"].value_counts())
print(f"\nProporción fraude: {df_check['FLAG'].mean()*100:.2f}%")
print("\nNulos por columna:")
print(df_check.isna().sum())

Verificación lectura: 31,188 filas × 16 columnas

Distribución FLAG:
FLAG
0    28779
1     2409
Name: count, dtype: int64

Proporción fraude: 7.72%

Nulos por columna:
CONS_NO                0
FLAG                   0
media                  0
std                    0
max                    0
min                    0
mediana                0
coef_variacion         0
skewness               0
kurtosis               0
dias_consumo_cero      0
racha_max_ceros        0
n_picos                0
n_valles               0
pct_dias_bajo_media    0
tendencia_lineal       0
dtype: int64


In [ ]:
print("="*60)
print("RESUMEN FEATURE ENGINEERING SGCC")
print("="*60)
print(f"Origen:       {RUTA_PROCESSED}")
print(f"Output:       {RUTA_CSV_FINAL}")
print(f"Clientes:     {df_pandas.shape[0]:,}")
print(f"Features:     {df_pandas.shape[1]-2} + CONS_NO + FLAG = {df_pandas.shape[1]} columnas")
print(f"Tamaño CSV:   {size_mb:.2f} MB")
print()
print("Distribución de la etiqueta:")
print(f"  Normales: {(df_pandas['FLAG']==0).sum():,} ({(df_pandas['FLAG']==0).mean()*100:.2f}%)")
print(f"  Fraude:   {(df_pandas['FLAG']==1).sum():,} ({(df_pandas['FLAG']==1).mean()*100:.2f}%)")
print()
print("PRÓXIMO PASO:")
print("Cargar el CSV en RapidMiner Studio")
print("Ruta Windows: volumes\\hdfs\\features\\sgcc_features.csv")
print("="*60)

df_enriquecido.unpersist()

RESUMEN FEATURE ENGINEERING SGCC
Origen:       /opt/spark-data/processed/sgcc_clean.parquet
Output:       /opt/spark-data/features/sgcc_features.csv
Clientes:     31,188
Features:     14 + CONS_NO + FLAG = 16 columnas
Tamaño CSV:   5.68 MB

Distribución de la etiqueta:
  Normales: 28,779 (92.28%)
  Fraude:   2,409 (7.72%)

PRÓXIMO PASO:
Cargar el CSV en RapidMiner Studio
Ruta Windows: volumes\hdfs\features\sgcc_features.csv
